# Supermarket Sales Analysis
**Author:** Chirag | Maharaja Ganga Singh University (MGSU), Bikaner
**Project:** IBM SkillsBuild — Data Analytics with AI Internship (AICTE | Bharat Cares)

This notebook analyzes 1,000 supermarket transactions across three branches to identify revenue drivers, customer behavior patterns, and actionable business recommendations.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})
PALETTE = ['#2E5EAA', '#F2A541', '#4CAF91', '#D9534F', '#8E6FCE', '#5BA3D0']
sns.set_palette(PALETTE)

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('SuperMarket_Analysis.csv')
df.columns = [c.strip() for c in df.columns]
df.head()

In [ ]:
df.info()

## 3. Data Cleaning

In [ ]:
# Check missing values and duplicates
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

df = df.drop_duplicates()
df['Date'] = pd.to_datetime(df['Date'])
df['Time'] = pd.to_datetime(df['Time'], format='%I:%M:%S %p', errors='coerce').dt.time
df['Hour'] = pd.to_datetime(df['Time'].astype(str)).dt.hour
df['Weekday'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()

weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['Weekday'] = pd.Categorical(df['Weekday'], categories=weekday_order, ordered=True)

df.describe()

## 4. Aggregation — Summary by Category

In [ ]:
branch_summary = df.groupby('Branch').agg(
    Total_Sales=('Sales','sum'), Orders=('Invoice ID','count'),
    Avg_Sale=('Sales','mean'), Avg_Rating=('Rating','mean'),
    Gross_Income=('gross income','sum')
).round(2).sort_values('Total_Sales', ascending=False)
branch_summary

In [ ]:
product_summary = df.groupby('Product line').agg(
    Total_Sales=('Sales','sum'), Orders=('Invoice ID','count'),
    Avg_Sale=('Sales','mean'), Gross_Income=('gross income','sum'),
    Avg_Rating=('Rating','mean')
).round(2).sort_values('Total_Sales', ascending=False)
product_summary

In [ ]:
customer_summary = df.groupby('Customer type').agg(
    Total_Sales=('Sales','sum'), Orders=('Invoice ID','count'),
    Avg_Sale=('Sales','mean'), Avg_Rating=('Rating','mean')
).round(2)
customer_summary

In [ ]:
payment_summary = df.groupby('Payment').agg(
    Total_Sales=('Sales','sum'), Orders=('Invoice ID','count')
).round(2).sort_values('Total_Sales', ascending=False)
payment_summary

## 5. Visualizations

In [ ]:
# Chart 1: Revenue by Branch
fig, ax = plt.subplots(figsize=(7,4.5))
branch_summary['Total_Sales'].plot(kind='bar', ax=ax, color=PALETTE[0])
ax.set_title('Total Revenue by Branch', fontweight='bold')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Revenue vs Gross Income by Product Line
fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(len(product_summary))
w = 0.38
ax.bar(x - w/2, product_summary['Total_Sales'], width=w, label='Revenue', color=PALETTE[0])
ax.bar(x + w/2, product_summary['Gross_Income']*5, width=w, label='Gross Income (x5 scaled)', color=PALETTE[1])
ax.set_xticks(x); ax.set_xticklabels(product_summary.index, rotation=30, ha='right')
ax.set_title('Revenue vs. Gross Income by Product Line', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Weekly revenue trend
weekday_sales = df.groupby('Weekday', observed=True)['Sales'].sum()
fig, ax = plt.subplots(figsize=(7,4.5))
weekday_sales.plot(kind='line', marker='o', ax=ax, color=PALETTE[2], linewidth=2.2)
ax.set_title('Revenue Pattern Across Days of the Week', fontweight='bold')
ax.set_ylabel('Revenue ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: Hourly revenue pattern
hourly_sales = df.groupby('Hour')['Sales'].sum()
fig, ax = plt.subplots(figsize=(7,4.5))
hourly_sales.plot(kind='bar', ax=ax, color=PALETTE[3])
ax.set_title('Revenue by Hour of Day', fontweight='bold')
ax.set_xlabel('Hour'); ax.set_ylabel('Revenue ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 5: Customer type revenue share
fig, ax = plt.subplots(figsize=(6,6))
vals = customer_summary['Total_Sales']
ax.pie(vals, labels=vals.index, autopct='%1.1f%%', colors=[PALETTE[0], PALETTE[1]],
       startangle=90, pctdistance=0.78, wedgeprops=dict(width=0.42, edgecolor='white'))
ax.set_title('Revenue Share: Member vs. Normal Customers', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 6: Order volume by payment method
fig, ax = plt.subplots(figsize=(7,4.5))
payment_summary['Orders'].plot(kind='barh', ax=ax, color=PALETTE[4])
ax.set_title('Order Volume by Payment Method', fontweight='bold')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Payment Method')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 7: Rating distribution by branch
fig, ax = plt.subplots(figsize=(7,4.5))
sns.boxplot(data=df, x='Branch', y='Rating', ax=ax, palette=PALETTE[:3])
ax.set_title('Customer Rating Distribution by Branch', fontweight='bold')
ax.set_xlabel('Branch')
ax.set_ylabel('Rating')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 8: Correlation heatmap
fig, ax = plt.subplots(figsize=(6,5))
num_cols = ['Unit price','Quantity','Sales','gross income','Rating']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Between Key Numeric Variables', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Business Insights

1. **Branch Performance:** Giza generates the highest revenue ($110,569) and highest average rating (7.07), while Cairo has the lowest rating (6.82).
2. **Category Leadership:** Food and beverages leads all categories in revenue and gross income; Health and beauty trails ~12% behind.
3. **Loyalty Program Impact:** Member customers drive 58.7% of total revenue and spend $29 more per order on average than Normal customers.
4. **Time-Based Demand:** Revenue peaks on Saturday and at 7 PM — key windows for staffing and inventory planning.
5. **Revenue Driver:** Quantity correlates strongly with Sales (r = 0.71); Rating shows almost no correlation with Sales (r ≈ 0.04).

## 7. Hypotheses

1. **H1 — Category Promotion:** A 4-week promotional campaign for Health and beauty will raise its revenue share by ≥10% next quarter.
2. **H2 — Service Training:** Targeted customer-service training at Cairo will raise its average rating closer to Giza's 7.07 benchmark within 3 months.
3. **H3 — Peak-Hour Staffing:** Increased staffing during 6–9 PM and Saturdays will improve fulfillment speed without a proportional cost increase.

## 8. Recommendations

1. **Investigate and Test Giza's Practices at Cairo** — The dataset does not record staffing, layout, or service-process details, so investigate Giza's actual practices on the ground, then pilot the most promising ones at Cairo for one quarter, tracking rating and revenue before and after.
2. **Invest in the Underperforming Category** — Allocate a defined marketing budget to Health and beauty for the next quarter (cross-promotions, better shelf placement) and track whether the revenue gap to the top category narrows.
3. **Expand the Loyalty Program** — Introduce a second loyalty tier with added benefits for high-spending Members, since they already outspend Normal customers by $29 per order on average.
4. **Optimize Staffing to Demand** — Align staff schedules with the observed demand peaks — Saturdays and the 6–9 PM window — rather than uniform shift patterns, to reduce wait times during the busiest hours.
5. **Track Satisfaction Independently** — Since rating and revenue are not correlated, track and report satisfaction as its own KPI on the dashboard rather than as a proxy for sales performance.